In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L1VzZXJzL2FydW5nb3BhbGFuL3JlcG9zL21vZm9ncmFwaHkvZG9jcw=='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/importlib/_bootstrap.py": 1756909683.181072, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/importlib/_bootstrap_external.py": 1756909683.1840236, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/zipimport.py": 1756909682.1390634, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/codecs.py": 1756909681.7841098, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/encodings/aliases.py": 1756909682.4869266, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/encodings/__init__.py": 1756909682.467157, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/encodings/utf_8.py": 1756909682.8301048, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/abc.py": 1756909681.751164, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/io.py": 1756909681.8854132, "/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/stat.py": 1756909682.0489361, "/Users/arungopalan/miniforge

In [2]:
import mofography as mgr
from ase.io import read
import numpy as np

# Step 1: load the structure
atoms = read("IRMOF-1.cif")
print(f"Loaded {atoms.get_chemical_formula()}, {len(atoms)} atoms")
print(f"Cell: {atoms.cell.cellpar()[:3].round(2)} Å")

Loaded C192H96O104Zn32, 424 atoms
Cell: [25.83 25.83 25.83] Å


/Users/arungopalan/miniforge3/envs/mgr_env/lib/python3.13/site-packages/ase/io/cif.py:411: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(


In [3]:
# Step 2: compute the distance grid (see distance_grid.qmd for details)
spacing = 0.5
dgrid = mgr.dgrid_from_atoms_rust(atoms, spacing=spacing)
print(f"Grid: {dgrid.shape}, void max: {dgrid.max():.1f} Å")

Grid: (52, 52, 52), void max: 7.5 Å


[RAYON] current_num_threads = 14


In [4]:
# Step 3: run periodic segmentation — one call replaces 5-7 manual steps
seg = mgr.do_periodic_segmentation_rust(
    dgrid, atoms,
    mask_thickness=1.0,        # void-space threshold (Å) — anything closer to framework is "wall"
    h=0.2,                     # h-maxima persistence (Å) — keep only peaks this tall
    min_distance=1.5,          # min separation between distinct maxima (Å)
    threshold_abs=1.5,         # absolute height threshold — ignore maxima shallower than this
    preserve_threshold=3.0,    # selective smoothing: don't smooth features below this (Å)
    smooth_threshold=6.0,      # selective smoothing: fully smooth features above this (Å)
    max_smooth_radius=2.0,     # selective smoothing: max Gaussian radius
    spacing=spacing,
    compute_connections=True,  # also detect windows between regions
    disable_tqdm=True,
)

[RAYON] current_num_threads = 14
/Users/arungopalan/repos/mofography/mofography/helper.py:546: RuntimeWarning: overflow encountered in scalar subtract
  pos_minus[dim] = max(pos_minus[dim] - 1, 0)


[pocket_maxima_check] Region 1: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


[pocket_maxima_check] Region 3: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


[pocket_maxima_check] Region 4: pocket_radius from maxima_info (5.284 Å) underestimates true region max (5.629 Å) by 0.345 Å; this suggests a missing or under-detected maximum.


[pocket_maxima_check] Region 5: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


[pocket_maxima_check] Region 9: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


... (4 outputs left out)

[pocket_maxima_check] Region 15: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


[pocket_maxima_check] Region 16: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


[pocket_maxima_check] Region 17: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


[pocket_maxima_check] Region 18: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.


In [5]:
# What did we get?
print(f"Pockets (PBC-grouped): {len(seg['maxima_pbc'])}")
print(f"All maxima (with periodic images): {len(seg['all_maxima'])}")
print(f"Unique regions influencing unit cell: {len(seg['unique_regions'])}")
print(f"Window connections: {len(seg['connections'])}")
print(f"PBC equivalence groups: {len(seg['pbc_groups'])}")

Pockets (PBC-grouped): 8
All maxima (with periodic images): 125
Unique regions influencing unit cell: 27
Window connections: 300
PBC equivalence groups: 8


In [6]:
maxima_pbc  = seg['maxima_pbc']   # one representative per unique pore
all_maxima  = seg['all_maxima']   # all periodic images
regions_pbc = seg['regions_pbc'] # the labeled voxel grid (PBC-merged)
pbc_groups  = seg['pbc_groups']   # which images belong to which pore

print(f"maxima_pbc  : {maxima_pbc.shape}  -> one entry per unique pore")
print(f"all_maxima  : {all_maxima.shape}  -> all periodic images")
print(f"regions_pbc : {regions_pbc.shape} -> labeled voxel grid")
print(f"pbc_groups  : {len(pbc_groups)} groups")

# Number of unique pores in this material
n_pores = int(regions_pbc.max())
print(f"\nNumber of unique pores: {n_pores}")

maxima_pbc  : (8, 3)  -> one entry per unique pore
all_maxima  : (125, 3)  -> all periodic images
regions_pbc : (52, 52, 52) -> labeled voxel grid
pbc_groups  : 8 groups

Number of unique pores: 8


In [7]:
# "maxima inside the unit cell" — same as close_or_inside_maxima[:regions.max()]
unit_cell_maxima = all_maxima[: int(regions_pbc.max())]
print(f"Maxima influencing the unit cell: {len(unit_cell_maxima)}")

Maxima influencing the unit cell: 8


In [8]:
# First three connections
for c in seg['connections'][:3]:
    print(c)

[array([49, 83], dtype=uint32) True 'periodic'
 array([[ 33.53192308, -12.916     ,  38.748     ]]) array([3.67783356])]
[array([ 5, 21], dtype=uint32) True 'internal'
 array([[7.20315385, 0.        , 0.        ]]) array([3.95680285])]
[array([41, 65], dtype=uint32) True 'periodic'
 array([[ 17.63530769,  38.748     , -12.916     ]]) array([3.67783356])]


In [9]:
# Interactive 3D — framework + regions + maxima
fig = mgr.show_mof_plotly(seg, atoms, framework_atoms=atoms, stylize=True)
fig.show()

[TIMING batched_mesh_for_atoms] 472 atoms, 1024 bonds | spheres: 0.00s, cylinders: 0.00s, concat: 0.00s | total faces: 212480


/Users/arungopalan/repos/mofography/mofography/visualize.py:2546: FutureWarning: Please use len(self) or, if your atoms are distributed, self.get_global_number_of_atoms.
  vertices_per_sphere = int(len(vertexcolors[vertexcolors!=bond_rgba_string])/ase_atoms.get_number_of_atoms())
/Users/arungopalan/repos/mofography/mofography/visualize.py:2547: FutureWarning: Please use len(self) or, if your atoms are distributed, self.get_global_number_of_atoms.
  vertex_atom_ids = np.repeat(np.arange(ase_atoms.get_number_of_atoms()), vertices_per_sphere)


No node props provided, using default set: [position, scaled_position, element and atomic_number]
No edge properties provided. Use the default set: [bond_order, bond_type]
isomorphic aromatic cycles is None, trying to calculate


In [10]:
import plotly.graph_objects as go
import plotly.express as px

cscale = px.colors.qualitative.Pastel
maxima = seg['all_maxima'][: int(seg['regions_pbc'].max())]

# Region isosurface, colored by pore label
surf = mgr.plot_surface_dgrid(
    dgrid, isovalue=1.5,
    ase_atoms=atoms,
    region_labels=seg['regions_pbc'],
    cscale=cscale,
    name='Pores (PBC merged)',
    stylize=True,
    opacity=0.8,
)

# Pore centers
pts = mgr.plot_maxima(dgrid, maxima, atoms, show_spheres=False, marker_colorscale=cscale)

# Cell box
cell = mgr.plot_cell(atoms)

# Framework mesh
mat_updated = mgr.update_with_covalent_bonds(atoms)
frame = mgr.plot_framework_as_mesh(mat_updated, stylize=True)

fig = go.Figure(
    data=[cell, surf, pts, *frame],
    layout=mgr.dark_layout,
)
fig.update_layout(
    scene_aspectmode='data',
    title=dict(text='IRMOF-1: segmented pores', x=0.5, xanchor='center', font=dict(color='white', size=16)),
)
fig.show()

[TIMING batched_mesh_for_atoms] 472 atoms, 1024 bonds | spheres: 0.00s, cylinders: 0.00s, concat: 0.00s | total faces: 212480


/Users/arungopalan/repos/mofography/mofography/visualize.py:2546: FutureWarning: Please use len(self) or, if your atoms are distributed, self.get_global_number_of_atoms.
  vertices_per_sphere = int(len(vertexcolors[vertexcolors!=bond_rgba_string])/ase_atoms.get_number_of_atoms())
/Users/arungopalan/repos/mofography/mofography/visualize.py:2547: FutureWarning: Please use len(self) or, if your atoms are distributed, self.get_global_number_of_atoms.
  vertex_atom_ids = np.repeat(np.arange(ase_atoms.get_number_of_atoms()), vertices_per_sphere)


No node props provided, using default set: [position, scaled_position, element and atomic_number]
No edge properties provided. Use the default set: [bond_order, bond_type]
isomorphic aromatic cycles is None, trying to calculate
